In [ ]:
import gurobipy as gb
import numpy as np
from grsc_cb_instance import GRSC_CB_Instance

# The GRSC-CB model
* Generalized Reserve Set Covering Problem with Buffer and Connectivity constraints
* It's based on the RSC, but with the definition of the **minimum quotas of ecological suitability**, of **buffer zones** and **connectivity constraints**

In [ ]:
instance = GRSC_CB_Instance()
grsc_cb = gb.Model("GRSC-CB")
grsc_cb.setParam('OutputFlag', 0)

Set parameter Username
Set parameter LicenseID to value 2796100
Academic license - for non-commercial use only - expires 2027-03-23


## The variables

### Land site

* $x_i$, $1 \leq i \leq |V|$ → binary variables defined as:

  $$
  \begin{align*}
  x_i =
  \begin{cases}
  1, & \text{if the land site $i \in V$ is selected}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$


In [ ]:
x = grsc_cb.addVars(instance.V, vtype=gb.GRB.BINARY, name="x")

### Core site

* $z_i$, $1 \leq i \leq |V|$ → binary variables defined as:

  $$
  \begin{align*}
  z_i =
  \begin{cases}
  1, & \text{if the land site $i \in V$ is part of the core area}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$

In [ ]:
z = grsc_cb.addVars(instance.V, vtype=gb.GRB.BINARY, name="z")

### Specie protection
* $u_s$, $1 \leq s \leq |S|$ → binary variables defined as:

  $$
  \begin{align*}
  u_i =
  \begin{cases}
  1, & \text{if the specie $s \in S$ is protected by the reserve}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$

In [ ]:
u = grsc_cb.addVars(instance.S, vtype=gb.GRB.BINARY, name="u")

### Connected components

* Let $y_i, 1 \leq i \leq |V|$ be an auxilary variable 
  $$
  \begin{align*}
  y_i =
  \begin{cases}
  1, & \text{if the land site $i \in V$ is connected to $r$ via an arc $(r,i)$}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$

In [ ]:
y = grsc_cb.addVars(instance.V, vtype=gb.GRB.BINARY, name="y")

## Objective function

Function that indicates the cost that needs to be **minimized**.
$$
\gamma(u,x,z) = \sum_{i\in V} c_i x_i
$$

In [ ]:
grsc_cb.setObjective(gb.quicksum(instance.c[i] * x[i] for i in instance.V), gb.GRB.MINIMIZE)

## Constraints

### S1-SQ

If a specie in $S_1$ is protected, the habitat suitability score for the land sites part of the core area must be at least the minimum quota of ecological suitability of that specie.
$$
\sum_{i\in V_s} w(i, s)z_i \geq \lambda_s u_s \quad \forall s \in S_1
$$

In [ ]:
for s in instance.S_1:
    grsc_cb.addConstr(gb.quicksum(instance.w[(i, s)] * z[i] for i in instance.v_s(s)) >= instance.lambda_s[s] * u[s], name="S1-SQ")


### S2-SQ

Similarly for a specie in $S_2$, considering land sites in the whole reserve.
$$
\sum_{i\in V_s} w(i, s)x_i \geq \lambda_s u_s \quad \forall s \in S_2
$$

In [ ]:
for s in instance.S_2:
    grsc_cb.addConstr(gb.quicksum(instance.w[(i, s)] * x[i] for i in instance.v_s(s)) >= instance.lambda_s[s] * u[s], name="S2-SQ")

### S1-PROTECT

At least $P_1$ species of $S_1$ must be protected. 
$$
\sum_{s\in S_1} u_s \geq P_1
$$

In [ ]:
grsc_cb.addConstr(gb.quicksum(u[s] for s in instance.S_1) >= instance.P_1, name="S1-PROTECT")

<gurobi.Constr *Awaiting Model Update*>

### S2-PROTECT

At least $P_2$ species of $S_2$ must be protected. 
$$
\sum_{s\in S_2} u_2 \geq P_2
$$

In [ ]:
grsc_cb.addConstr(gb.quicksum(u[s] for s in instance.S_2) >= instance.P_2, name="S2-PROTECT")

<gurobi.Constr *Awaiting Model Update*>

### LINK

If a site is in the core, then it must also be in the reserve.
$$
z_i \leq x_i \quad \forall i \in V
$$

In [ ]:
for i in instance.V:
    grsc_cb.addConstr(z[i] <= x[i], name=f"LINK")   

---
#### d-neighborhood

For $d \geq 0, d \in \mathbb{N}, i \in V$ the $d$-neighborhood is defined as the set of sites at distance $d$ from $i$.
$$
\begin{align*}

\delta_d (i) = \{j \in V_{i  \neq j} | \text{the number of jumps from $i$ and $j$ is max $d$} \}

\end{align*}
$$
$\delta_d^+(i)$ it the d-neighborhood that contains also $i$.

### d-BUFF.1

Every core site is surrounded by a buffer area of normal sites in its $d$-neighborhood
$$
z_i \leq x_j, \quad \forall j \in \delta_d(i), \quad \forall i \in V
$$

In [ ]:
for i in instance.V:
    for j in instance.delta_d(i):
        grsc_cb.addConstr(z[i] <= x[j], name=f"d-BUFF.1")

### d-BUFF.2

Every normal site must have at least one core site in the $d$-neighborhood
$$
x_i \leq \sum_{j \in \delta_d(i)} z_j \quad \forall i \in V
$$

In [ ]:
for i in instance.V:  
    grsc_cb.addConstr(x[i] <= gb.quicksum(z[j] for j in instance.delta_d(i)), name=f"d-BUFF.2")

---
### NCOMP

There's a max of $k$ connected components in the reserve.
$$
\sum_{j\in V} y_j \leq k
$$

In [ ]:
grsc_cb.addConstr(gb.quicksum(y[j] for j in instance.V) <= instance.k, name=f"NCOMP")

### YZ

If a site $i$ is connected to the root, then it must be considered in the core area.
$$
y_i \leq z_i \quad \forall i \in V
$$

In [ ]:
for i in instance.V:
    grsc_cb.addConstr(y[i] <= z[i], name=f"YZ")

For the connectivity constraint:

#### r-arc-node-separator
* A root $r$ is added to the graph: we can consider $G_r=(V_r. E_r)$ where $V_r= V \cup \{r\}$ and $E_r = E \cup \{(r,i)|i\in V\}$ (so all the nodes from the original graph are connected to $r$)
* The arcs that connect the nodes to the root $r$ are called **$r$-arcs** and the set of $r$-arcs is called $A_r$
* Given $l \in V$, an **$r$-arc-node-separator** is a tuple $W=(W_V, W_A)$, where $W_V \subseteq V$ and $W_A \subseteq A_r$, such that  if $W$ is removed from $G_r$ then the site $l$ can't be reached by $r$
* $W_l$ is the set of all possible $r$-arc-node-separators w.r.t $l$
* to have connectivity, we want in the solution to have a path from $r$ to all nodes present in the solution

### CORECON

If the land site $l$ is in the core area of the solution, then for all the $W \in W_l$, I must have at least one node from $W_V$ or one arc from $W_A$. 
$$
\sum_{i\in W_V} z_i + \sum_{j \in W_A } y_j \geq z_l, \quad \forall W \in W_l, \quad \forall l \in V
$$

Since the number of these constraints is exponential, a branch-and-cut framework is developed.